In [2]:
%pip install sqlalchemy pymysql

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path
from getpass import getpass

import pandas as pd

from sqlalchemy import (
    URL,
    create_engine,
    text
)

In [4]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

cleaned_dir = project_root / "data" / "cleaned"

print("Project root:", project_root)
print("Cleaned directory:", cleaned_dir)

Project root: c:\Users\visva\Desktop\Folder\Document\GUVI\Projects\P1
Cleaned directory: c:\Users\visva\Desktop\Folder\Document\GUVI\Projects\P1\data\cleaned


In [5]:
files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation":
        "product_category_name_translation.csv"
}

In [6]:
data_types = {
    "customers": {
        "customer_zip_code_prefix": "string"
    },
    "sellers": {
        "seller_zip_code_prefix": "string"
    },
    "geolocation": {
        "geolocation_zip_code_prefix": "string"
    }
}

In [7]:
date_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": [
        "shipping_limit_date"
    ],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

In [8]:
data = {}

for table_name, file_name in files.items():
    data[table_name] = pd.read_csv(
        cleaned_dir / file_name,
        dtype=data_types.get(table_name),
        parse_dates=date_columns.get(table_name)
    )

    print(
        table_name,
        data[table_name].shape
    )

customers (99441, 5)
geolocation (738332, 5)
order_items (112650, 7)
payments (103886, 5)
reviews (99224, 7)
orders (99441, 8)
products (32951, 9)
sellers (3095, 4)
category_translation (71, 2)


In [9]:
print(
    data["customers"][
        "customer_zip_code_prefix"
    ].str.len().value_counts()
)

customer_zip_code_prefix
5    99441
Name: count, dtype: Int64


In [10]:
mysql_username = input(
    "MySQL username [root]: "
).strip() or "root"

mysql_password = getpass(
    "MySQL password: "
)

connection_url = URL.create(
    drivername="mysql+pymysql",
    username=mysql_username,
    password=mysql_password,
    host="localhost",
    port=3306,
    database="cart2insights"
)

engine = create_engine(
    connection_url,
    pool_pre_ping=True
)

In [11]:
with engine.connect() as connection:
    database_name = connection.execute(
        text("SELECT DATABASE()")
    ).scalar()

    print("Connected to:", database_name)

Connected to: cart2insights


In [12]:
load_order = [
    "category_translation",
    "customers",
    "products",
    "sellers",
    "orders",
    "order_items",
    "payments",
    "reviews",
    "geolocation"
]

In [13]:
for table_name in load_order:
    df = data[table_name]

    with engine.connect() as connection:
        existing_rows = connection.execute(
            text(
                f"SELECT COUNT(*) "
                f"FROM `{table_name}`"
            )
        ).scalar()

    if existing_rows == len(df):
        print(
            f"Skipped {table_name}: "
            f"already contains {existing_rows} rows"
        )
        continue

    if existing_rows > 0:
        raise RuntimeError(
            f"{table_name} contains {existing_rows} rows. "
            "The table may be partially loaded."
        )

    print(
        f"Loading {table_name}: "
        f"{len(df)} rows..."
    )

    with engine.begin() as connection:
        df.to_sql(
            name=table_name,
            con=connection,
            if_exists="append",
            index=False,
            chunksize=2000,
            method="multi"
        )

    print(f"Completed: {table_name}")

Skipped category_translation: already contains 71 rows
Skipped customers: already contains 99441 rows
Skipped products: already contains 32951 rows
Skipped sellers: already contains 3095 rows
Skipped orders: already contains 99441 rows
Skipped order_items: already contains 112650 rows
Skipped payments: already contains 103886 rows
Skipped reviews: already contains 99224 rows
Skipped geolocation: already contains 738332 rows


In [14]:
verification = []

with engine.connect() as connection:
    for table_name in load_order:
        database_rows = connection.execute(
            text(
                f"SELECT COUNT(*) "
                f"FROM `{table_name}`"
            )
        ).scalar()

        csv_rows = len(data[table_name])

        verification.append({
            "table": table_name,
            "csv_rows": csv_rows,
            "database_rows": database_rows,
            "rows_match": csv_rows == database_rows
        })

verification_df = pd.DataFrame(verification)
verification_df

,table,csv_rows,database_rows,rows_match
0,category_translation,71,71,True
1,customers,99441,99441,True
2,products,32951,32951,True
3,sellers,3095,3095,True
4,orders,99441,99441,True
5,order_items,112650,112650,True
6,payments,103886,103886,True
7,reviews,99224,99224,True
8,geolocation,738332,738332,True


In [15]:
relationship_queries = {
    "orders_without_customer": """
        SELECT COUNT(*)
        FROM orders o
        LEFT JOIN customers c
            ON o.customer_id = c.customer_id
        WHERE c.customer_id IS NULL
    """,

    "items_without_order": """
        SELECT COUNT(*)
        FROM order_items oi
        LEFT JOIN orders o
            ON oi.order_id = o.order_id
        WHERE o.order_id IS NULL
    """,

    "items_without_product": """
        SELECT COUNT(*)
        FROM order_items oi
        LEFT JOIN products p
            ON oi.product_id = p.product_id
        WHERE p.product_id IS NULL
    """,

    "items_without_seller": """
        SELECT COUNT(*)
        FROM order_items oi
        LEFT JOIN sellers s
            ON oi.seller_id = s.seller_id
        WHERE s.seller_id IS NULL
    """
}

with engine.connect() as connection:
    for check_name, query in relationship_queries.items():
        result = connection.execute(
            text(query)
        ).scalar()

        print(check_name, ":", result)

orders_without_customer : 0
items_without_order : 0
items_without_product : 0
items_without_seller : 0


In [16]:
from sqlalchemy import text

try:
    with engine.connect() as connection:
        database_name = connection.execute(
            text("SELECT DATABASE()")
        ).scalar()

        print("Connected successfully")
        print("Current database:", database_name)

except Exception as error:
    print("Connection failed:")
    print(error)

Connected successfully
Current database: cart2insights


In [17]:
mysql_password = getpass("Enter your MySQL password: ")

In [18]:
from pathlib import Path
import pandas as pd

project_dir = Path.cwd()

# Handles both project-root and notebooks working directories
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

cleaned_dir = project_dir / "data" / "cleaned"

print("Cleaned folder:", cleaned_dir)
print("Available files:", [file.name for file in cleaned_dir.glob("*.csv")])

Cleaned folder: c:\Users\visva\Desktop\Folder\Document\GUVI\Projects\P1\data\cleaned
Available files: ['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


In [20]:
customers = pd.read_csv(
    cleaned_dir / "olist_customers_dataset.csv",
    dtype={
        "customer_id": "string",
        "customer_unique_id": "string",
        "customer_zip_code_prefix": "string"
    }
)

print("Pandas rows:", len(customers))
customers.head()

Pandas rows: 99441


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [22]:
customers.to_sql(
    name="olist_customers_dataset.csv",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=5000
)

print("Customers table loaded successfully")

Customers table loaded successfully


In [23]:
from sqlalchemy import text

with engine.connect() as connection:
    mysql_rows = connection.execute(
        text("SELECT COUNT(*) FROM customers")
    ).scalar()

print("Pandas rows:", len(customers))
print("MySQL rows:", mysql_rows)
print("Counts match:", len(customers) == mysql_rows)

Pandas rows: 99441
MySQL rows: 99441
Counts match: True


In [40]:
files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

In [ ]:
parent_files = {
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv"
}

In [ ]:
parent_tables = {}

for table_name, file_name in parent_files.items():
    file_path = cleaned_dir / file_name

    parent_tables[table_name] = pd.read_csv(
        file_path,
        dtype={
            "geolocation_zip_code_prefix": "string",
            "seller_zip_code_prefix": "string",
            "product_id": "string",
            "seller_id": "string"
        }
    )

    print(table_name, parent_tables[table_name].shape)

geolocation (738332, 5)
category_translation (71, 2)
products (32951, 9)
sellers (3095, 4)


In [45]:
from sqlalchemy import inspect, text

def load_and_verify(table_name, dataframe, engine):
    inspector = inspect(engine)
    table_exists = table_name in inspector.get_table_names()

    if table_exists:
        with engine.connect() as connection:
            existing_rows = connection.execute(
                text(f"SELECT COUNT(*) FROM `{table_name}`")
            ).scalar()
    else:
        existing_rows = 0

    if not table_exists:
        dataframe.to_sql(
            name=table_name,
            con=engine,
            if_exists="append",
            index=False,
            chunksize=5000
        )

        print(f"{table_name}: table created and data loaded")

    elif existing_rows == 0:
        dataframe.to_sql(
            name=table_name,
            con=engine,
            if_exists="append",
            index=False,
            chunksize=5000
        )

        print(f"{table_name}: data loaded into existing table")

    elif existing_rows == len(dataframe):
        print(
            f"{table_name}: already loaded "
            f"({existing_rows} rows), skipping"
        )

    else:
        print(
            f"{table_name}: stopped because MySQL already contains "
            f"{existing_rows} rows, but Pandas contains {len(dataframe)} rows"
        )
        return

    with engine.connect() as connection:
        mysql_rows = connection.execute(
            text(f"SELECT COUNT(*) FROM `{table_name}`")
        ).scalar()

    print(
        f"{table_name}: "
        f"Pandas={len(dataframe)}, "
        f"MySQL={mysql_rows}, "
        f"Match={len(dataframe) == mysql_rows}"
    )

In [46]:
for table_name, dataframe in parent_tables.items():
    load_and_verify(table_name, dataframe, engine)

geolocation: already loaded (738332 rows), skipping
geolocation: Pandas=738332, MySQL=738332, Match=True
category_translation: already loaded (71 rows), skipping
category_translation: Pandas=71, MySQL=71, Match=True
products: already loaded (32951 rows), skipping
products: Pandas=32951, MySQL=32951, Match=True
sellers: already loaded (3095 rows), skipping
sellers: Pandas=3095, MySQL=3095, Match=True


In [47]:
orders = pd.read_csv(
    cleaned_dir / "olist_orders_dataset.csv",
    dtype={
        "order_id": "string",
        "customer_id": "string"
    }
)

In [48]:
order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in order_date_columns:
    orders[column] = pd.to_datetime(
        orders[column],
        errors="coerce"
    )

In [49]:
print("Orders shape:", orders.shape)
orders.info()

Orders shape: (99441, 8)
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  string        
 1   customer_id                    99441 non-null  string        
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(1), string(2)
memory usage: 13.0 MB


In [50]:
load_and_verify(
    table_name="orders",
    dataframe=orders,
    engine=engine
)

orders: already loaded (99441 rows), skipping
orders: Pandas=99441, MySQL=99441, Match=True


In [51]:
with engine.connect() as connection:
    invalid_customers = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM orders AS o
            LEFT JOIN customers AS c
                ON o.customer_id = c.customer_id
            WHERE c.customer_id IS NULL
        """)
    ).scalar()

print("Orders with invalid customer IDs:", invalid_customers)

Orders with invalid customer IDs: 0


In [52]:
order_items = pd.read_csv(
    cleaned_dir / "olist_order_items_dataset.csv",
    dtype={
        "order_id": "string",
        "product_id": "string",
        "seller_id": "string"
    }
)

payments = pd.read_csv(
    cleaned_dir / "olist_order_payments_dataset.csv",
    dtype={
        "order_id": "string"
    }
)

reviews = pd.read_csv(
    cleaned_dir / "olist_order_reviews_dataset.csv",
    dtype={
        "review_id": "string",
        "order_id": "string"
    }
)

In [53]:
order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for column in review_date_columns:
    reviews[column] = pd.to_datetime(
        reviews[column],
        errors="coerce"
    )

In [54]:
print("Order items:", order_items.shape)
print("Payments:", payments.shape)
print("Reviews:", reviews.shape)

Order items: (112650, 7)
Payments: (103886, 5)
Reviews: (99224, 7)


In [55]:
transaction_tables = {
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews
}

for table_name, dataframe in transaction_tables.items():
    load_and_verify(
        table_name,
        dataframe,
        engine
    )

order_items: already loaded (112650 rows), skipping
order_items: Pandas=112650, MySQL=112650, Match=True
payments: already loaded (103886 rows), skipping
payments: Pandas=103886, MySQL=103886, Match=True
reviews: already loaded (99224 rows), skipping
reviews: Pandas=99224, MySQL=99224, Match=True


In [56]:
all_dataframes = {
    "customers": customers,
    **parent_tables,
    "orders": orders,
    **transaction_tables
}

verification_results = []

for table_name, dataframe in all_dataframes.items():
    with engine.connect() as connection:
        mysql_rows = connection.execute(
            text(f"SELECT COUNT(*) FROM `{table_name}`")
        ).scalar()

    verification_results.append({
        "table": table_name,
        "pandas_rows": len(dataframe),
        "mysql_rows": mysql_rows,
        "match": len(dataframe) == mysql_rows
    })

verification_df = pd.DataFrame(verification_results)
verification_df

,table,pandas_rows,mysql_rows,match
0,customers,99441,99441,True
1,geolocation,738332,738332,True
2,category_translation,71,71,True
3,products,32951,32951,True
4,sellers,3095,3095,True
5,orders,99441,99441,True
6,order_items,112650,112650,True
7,payments,103886,103886,True
8,reviews,99224,99224,True


In [57]:
relationship_queries = {
    "orders → customers": """
        SELECT COUNT(*)
        FROM orders o
        LEFT JOIN customers c
            ON o.customer_id = c.customer_id
        WHERE c.customer_id IS NULL
    """,

    "order_items → orders": """
        SELECT COUNT(*)
        FROM order_items oi
        LEFT JOIN orders o
            ON oi.order_id = o.order_id
        WHERE o.order_id IS NULL
    """,

    "order_items → products": """
        SELECT COUNT(*)
        FROM order_items oi
        LEFT JOIN products p
            ON oi.product_id = p.product_id
        WHERE p.product_id IS NULL
    """,

    "order_items → sellers": """
        SELECT COUNT(*)
        FROM order_items oi
        LEFT JOIN sellers s
            ON oi.seller_id = s.seller_id
        WHERE s.seller_id IS NULL
    """,

    "payments → orders": """
        SELECT COUNT(*)
        FROM payments pay
        LEFT JOIN orders o
            ON pay.order_id = o.order_id
        WHERE o.order_id IS NULL
    """,

    "reviews → orders": """
        SELECT COUNT(*)
        FROM reviews r
        LEFT JOIN orders o
            ON r.order_id = o.order_id
        WHERE o.order_id IS NULL
    """
}

relationship_results = []

with engine.connect() as connection:
    for relationship, query in relationship_queries.items():
        invalid_rows = connection.execute(
            text(query)
        ).scalar()

        relationship_results.append({
            "relationship": relationship,
            "invalid_rows": invalid_rows,
            "valid": invalid_rows == 0
        })

relationship_df = pd.DataFrame(relationship_results)
relationship_df

,relationship,invalid_rows,valid
0,orders → customers,0,True
1,order_items → orders,0,True
2,order_items → products,0,True
3,order_items → sellers,0,True
4,payments → orders,0,True
5,reviews → orders,0,True


In [58]:
mysql_password = getpass("Enter your MySQL password: ")